In [1]:
!pip install scikit-learn==1.8.0
import pandas as pd
import numpy as np

import pickle, os

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, \
                            classification_report, confusion_matrix

RANDOM_SEED = 42

from google.colab import drive
drive.mount('/content/drive')
df = pd.read_excel('/content/drive/MyDrive/MINI_Project_BATCH8/Datasets/liver_transplant_clinical_dataset.xlsx')

TARGET = 'Match Label'

X = df.drop(TARGET, axis=1).copy()
y = df[TARGET].astype(int)

print(f"Features: {X.shape[1]}  |  Samples: {len(y)}")
print(f"Class distribution: {dict(y.value_counts().sort_index())}")

# ── Detect numeric vs categorical columns ─────────────────────────────────────
cat_cols, num_cols = [], []
for col in X.columns:
    try:
        pd.to_numeric(X[col].dropna(), errors='raise')
        num_cols.append(col)
    except:
        cat_cols.append(col)

for col in num_cols:
    X[col] = pd.to_numeric(X[col], errors='coerce').astype('float64')
for col in cat_cols:
    X[col] = X[col].fillna('Unknown').astype(str)

# ── Impute & encode ───────────────────────────────────────────────────────────
num_imputer = SimpleImputer(strategy='median')
X[num_cols] = num_imputer.fit_transform(X[num_cols])

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

X = X.astype('float64')

# ── Train / test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y)

# ── Model ─────────────────────────────────────────────────────────────────────
model = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    min_samples_leaf=15,
    random_state=RANDOM_SEED
)

# ── Cross-validation ──────────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_res = cross_validate(model, X_train.values, y_train, cv=cv,
                        scoring=['accuracy', 'f1', 'roc_auc'], n_jobs=-1)

print(f"\nCV Accuracy : {cv_res['test_accuracy'].mean()*100:.2f}%")
print(f"CV F1 Score : {cv_res['test_f1'].mean():.4f}")
print(f"CV ROC-AUC  : {cv_res['test_roc_auc'].mean():.4f}")

# ── Train final model ─────────────────────────────────────────────────────────
model.fit(X_train.values, y_train)

# ── Evaluate ──────────────────────────────────────────────────────────────────
y_pred = model.predict(X_test.values)
y_prob = model.predict_proba(X_test.values)[:, 1]

train_acc = accuracy_score(y_train, model.predict(X_train.values))
test_acc  = accuracy_score(y_test, y_pred)
overfit   = train_acc - cv_res['test_accuracy'].mean()

print(f"\nTrain Accuracy : {train_acc*100:.2f}%")
print(f"Test  Accuracy : {test_acc*100:.2f}%")
print(f"Overfit Gap    : {overfit*100:+.2f}%  (target: < 5%)")
print(f"Test ROC-AUC   : {roc_auc_score(y_test, y_prob):.4f}")
print(f"\nClassification Report:\n")
print(classification_report(y_test, y_pred,
      target_names=["Not Recommended", "Suitable Match"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ── Feature importances ───────────────────────────────────────────────────────
fi = pd.DataFrame({'Feature': X.columns,
                   'Importance': model.feature_importances_}
                 ).sort_values('Importance', ascending=False)
print("\nTop 10 Feature Importances:")
print(fi.head(10).to_string(index=False))

filename = 'liver_transplant.sav'
pickle.dump(model, open(filename, 'wb'))
print(f"\nModel saved → {filename}")


meta = {
    'model':          model,
    'num_imputer':    num_imputer,
    'label_encoders': label_encoders,
    'feature_names':  list(X.columns),
    'cat_cols':       cat_cols,
    'num_cols':       num_cols,
}
with open('liver_model_meta.pkl', 'wb') as f:
    pickle.dump(meta, f)
print("Metadata saved → liver_model_meta.pkl")

# ── Quick prediction test ─────────────────────────────────────────────────────
loaded_model = pickle.load(open('liver_transplant.sav', 'rb'))

def predict_match(sample_row):
    row = sample_row.copy()
    pred = loaded_model.predict([row])[0]
    return "Matched" if pred == 1 else "Unmatched"

example = X_test.iloc[1].copy()
prediction = predict_match(example)
print("Example Prediction:", prediction)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 24.1 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
Mounted at /content/drive
Features: 31  |  Samples: 8000
Class distribution: {0: np.int64(4246), 1: np.int64(3754)}

CV Accuracy : 93.41%
CV F1 Score : 0.9318
CV ROC-AUC  : 0.9864

Train Accuracy : 97.09%
Test  Accuracy : 94.06%
Overfit Gap    : +3.69%  (target: < 5%)
Test ROC-AUC   : 0.9878

Classification Report:

                 precision    recall  f1-score   support

Not Recommended       0.96      0.92      0.94       849
 Suitable Match       0.92      0.96      0.94       751

       accuracy                           0.94      1600
      macro avg       0.94      0.94      0.94      1600
   weighted avg       0.94      0.94      0.94      1600

Confusion Matrix:
[[783  66]
 [ 29 722]]

Top 10 Feature Importances:
              

In [2]:
from importlib.metadata import version
print(version('scikit-learn'))
print(version('numpy'))

1.8.0
2.0.2
